[![Labellerr](https://storage.googleapis.com/labellerr-cdn/%200%20Labellerr%20template/notebook.webp)](https://www.labellerr.com)

# **Distillation Learning Pipeline**

---

[![labellerr](https://img.shields.io/badge/Labellerr-BLOG-black.svg)](https://www.labellerr.com/blog)
[![Youtube](https://img.shields.io/badge/Labellerr-YouTube-b31b1b.svg)](https://www.youtube.com/@Labellerr)
[![Github](https://img.shields.io/badge/Labellerr-GitHub-green.svg)](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)

This notebook demonstrates a knowledge distillation pipeline where Claude Opus acts as a "teacher" model to generate training data. It crafts 50 prompts requesting single-file HTML/CSS landing pages for various businesses, collecting 45 high-quality responses from Claude. This dataset is then used to fine-tune Qwen2.5-1.5B — a tiny student model — using LoRA (adapting only 0.28% of parameters) and 4-bit quantization via Unsloth. After 3 training epochs, the fine-tuned model can generate landing page code on its own. The core idea: use an expensive, powerful model to teach a cheap, small one to perform a specific task.

## SETTING-UP THE ENVIRONMENT

In [ ]:
# !pip install -q anthropic transformers trl peft bitsandbytes accelerate datasets sentencepiece tqdm torch

In [ ]:
# !pip install "unsloth @ git+https://github.com/unslothai/unsloth.git" -q
# !pip install "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git" -q

In [16]:
!nvidia-smi

Mon Mar  2 07:42:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P0             31W /   70W |     155MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:
import torch
print(torch.cuda.device_count())  # if 2, you can use both
print(torch.cuda.get_device_properties(0).total_memory / 1e9)

2
15.637086208


In [19]:
import anthropic
import json
from IPython.display import display, Markdown

import os
import time
import torch
from unsloth import FastLanguageModel

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from peft import prepare_model_for_kbit_training
from trl import SFTTrainer
import gc
import tqdm

In [20]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Supported architectures: {torch.cuda.get_arch_list()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.9.0+cu126
CUDA available: True
CUDA version: 12.6
Supported architectures: ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']
GPU name: Tesla T4


## SET-UP ENVIRONMENT VARIABLE

In [21]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
CLAUDE_API = user_secrets.get_secret("CLAUDE_API")
HUGGINGFACE_TOKEN = user_secrets.get_secret("HUGGINGFACE_TOKEN")


## SET-UP CLAUDE MODEL API

In [22]:
client = anthropic.Anthropic(api_key=str(CLAUDE_API))

In [23]:
# List all available models
models = client.models.list()

for model in models.data:
    print(f"ID: {model.id}")
    print(f"Name: {model.display_name}")
    print(f"Created: {model.created_at}")
    print("---")

ID: claude-sonnet-4-6
Name: Claude Sonnet 4.6
Created: 2026-02-17 00:00:00+00:00
---
ID: claude-opus-4-6
Name: Claude Opus 4.6
Created: 2026-02-04 00:00:00+00:00
---
ID: claude-opus-4-5-20251101
Name: Claude Opus 4.5
Created: 2025-11-24 00:00:00+00:00
---
ID: claude-haiku-4-5-20251001
Name: Claude Haiku 4.5
Created: 2025-10-15 00:00:00+00:00
---
ID: claude-sonnet-4-5-20250929
Name: Claude Sonnet 4.5
Created: 2025-09-29 00:00:00+00:00
---
ID: claude-opus-4-1-20250805
Name: Claude Opus 4.1
Created: 2025-08-05 00:00:00+00:00
---
ID: claude-opus-4-20250514
Name: Claude Opus 4
Created: 2025-05-22 00:00:00+00:00
---
ID: claude-sonnet-4-20250514
Name: Claude Sonnet 4
Created: 2025-05-22 00:00:00+00:00
---
ID: claude-3-haiku-20240307
Name: Claude Haiku 3
Created: 2024-03-07 00:00:00+00:00
---


In [24]:
response = client.messages.create(
    model="claude-haiku-4-5-20251001",  # fast & cheap for testing
    max_tokens=256,
    messages=[
        {"role": "user", "content": "Say hello in 3 different languages."}
    ]
)

display(Markdown(response.content[0].text))

# Hello in 3 Languages

1. **English:** Hello

2. **Spanish:** Hola

3. **French:** Bonjour

## CREATE LIST OF PROMPTS FOR DISTILLATION

In [25]:
prompts = [
    "Build a single-file with HTML and CSS for a landing page for a SaaS AI automation platform.",
    "Build a single-file with HTML and CSS for a landing page for a fintech mobile banking app.",
    "Build a single-file with HTML and CSS for a landing page for a cryptocurrency exchange.",
    "Build a single-file with HTML and CSS for a landing page for a Web3 NFT marketplace.",
    "Build a single-file with HTML and CSS for a landing page for a cloud storage service.",
    "Build a single-file with HTML and CSS for a landing page for a cybersecurity company.",
    "Build a single-file with HTML and CSS for a landing page for a data analytics dashboard tool.",
    "Build a single-file with HTML and CSS for a landing page for a no-code website builder.",
    "Build a single-file with HTML and CSS for a landing page for a developer API platform.",
    "Build a single-file with HTML and CSS for a landing page for a DevOps automation tool.",
    "Build a single-file with HTML and CSS for a landing page for a productivity task management app.",
    "Build a single-file with HTML and CSS for a landing page for a remote team collaboration tool.",
    "Build a single-file with HTML and CSS for a landing page for a startup incubator program.",
    "Build a single-file with HTML and CSS for a landing page for an AI-powered chatbot service.",
    "Build a single-file with HTML and CSS for a landing page for a personal finance tracking app.",
    "Build a single-file with HTML and CSS for a landing page for an online course platform.",
    "Build a single-file with HTML and CSS for a landing page for a coding bootcamp.",
    "Build a single-file with HTML and CSS for a landing page for a fitness coaching program.",
    "Build a single-file with HTML and CSS for a landing page for a mental health support app.",
    "Build a single-file with HTML and CSS for a landing page for a telemedicine healthcare service.",
    "Build a single-file with HTML and CSS for a landing page for an e-commerce fashion brand.",
    "Build a single-file with HTML and CSS for a landing page for a dropshipping business.",
    "Build a single-file with HTML and CSS for a landing page for a sustainable eco-friendly product brand.",
    "Build a single-file with HTML and CSS for a landing page for a luxury real estate agency.",
    "Build a single-file with HTML and CSS for a landing page for a property rental platform.",
    "Build a single-file with HTML and CSS for a landing page for a travel booking website.",
    "Build a single-file with HTML and CSS for a landing page for a digital marketing agency.",
    "Build a single-file with HTML and CSS for a landing page for a social media management tool.",
    "Build a single-file with HTML and CSS for a landing page for an influencer marketing platform.",
    "Build a single-file with HTML and CSS for a landing page for a YouTube automation service.",
    "Build a single-file with HTML and CSS for a landing page for a podcast hosting platform.",
    "Build a single-file with HTML and CSS for a landing page for a video editing SaaS tool.",
    "Build a single-file with HTML and CSS for a landing page for a graphic design subscription service.",
    "Build a single-file with HTML and CSS for a landing page for a print-on-demand merchandise brand.",
    "Build a single-file with HTML and CSS for a landing page for a subscription box service.",
    "Build a single-file with HTML and CSS for a landing page for a food delivery startup.",
    "Build a single-file with HTML and CSS for a landing page for a restaurant reservation app.",
    "Build a single-file with HTML and CSS for a landing page for a local home services marketplace.",
    "Build a single-file with HTML and CSS for a landing page for a job recruitment platform.",
    "Build a single-file with HTML and CSS for a landing page for a resume builder tool.",
    "Build a single-file with HTML and CSS for a landing page for a freelance marketplace.",
    "Build a single-file with HTML and CSS for a landing page for a car rental service.",
    "Build a single-file with HTML and CSS for a landing page for an electric vehicle charging network.",
    "Build a single-file with HTML and CSS for a landing page for a gaming community platform.",
    "Build a single-file with HTML and CSS for a landing page for a mobile game launch.",
    "Build a single-file with HTML and CSS for a landing page for a VR fitness app.",
    "Build a single-file with HTML and CSS for a landing page for an AI-powered resume screening tool.",
    "Build a single-file with HTML and CSS for a landing page for a blockchain development agency.",
    "Build a single-file with HTML and CSS for a landing page for a robotics automation startup.",
    "Build a single-file with HTML and CSS for a landing page for a space-tech satellite data company."
]

## EXTRACT PROMPT RESPONSE FROM CLAUDE

In [ ]:
SYSTEM_PROMPT = """You are an expert web developer. When asked to build a webpage, 
return a complete single HTML file with embedded CSS.
Be in single file and, use CSS to make it visually appealing, no comments it.
The entire response must be under 7000 tokens."""

dataset = []

for i, prompt in enumerate(prompts, 1):
    response = client.messages.create(
        model="claude-opus-4-5-20251101",
        max_tokens=8096,
        messages=[{"role": "user", "content": prompt}],
        system=SYSTEM_PROMPT
    )

    # ── Token usage for this prompt ──────────────────────
    usage        = response.usage
    input_tok    = usage.input_tokens
    output_tok   = usage.output_tokens
    total_tok    = input_tok + output_tok

    print(f"\n[{i}/{len(prompts)}] {prompt[:55]}")
    print(f"  Stop reason : {response.stop_reason}")
    print(f"  Tokens      : {input_tok} in / {output_tok} out / {total_tok} total")

    if response.stop_reason == "max_tokens":
        print(f"  ❌ Skipped (truncated)")
        continue

    answer = response.content[0].text
    dataset.append({"instruction": prompt, "output": answer})

    with open("webpage_dataset_2.json", "w") as f:
        json.dump(dataset, f, indent=2)

    print(f"✅ Saved ({len(answer)} chars)")

## LOGIN IN HUGGINGFACE

In [26]:
from huggingface_hub import login, whoami

login(HUGGINGFACE_TOKEN)
print("Token Loaded Successfully")
print(whoami()["name"])

Token Loaded Successfully
yashsuman


## LOAD THE MODEL USING UNSLOTH

In [27]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=7000,       # supports long sequences natively
    dtype=None,                # auto-detects best dtype for your GPU
    load_in_4bit=True,         # 4-bit quantization
)

==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## SET-UP LORA FINTUNING USING UNSLOTH

In [28]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # ← unsloth's optimized checkpointing
    random_state=42,
)

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 0 MLP layers.


In [29]:
from datasets import Dataset
import json

with open("/kaggle/input/datasets/yashsuman/claude-prompt-result/webpage_dataset_2.json") as f:
    dataset = json.load(f)

formatted = [
    {
        "text": f"<|im_start|>user\n{item['instruction']}<|im_end|>\n<|im_start|>assistant\n{item['output']}<|im_end|>"
    }
    for item in dataset
]

hf_dataset = Dataset.from_list(formatted)
print(f"✅ {len(hf_dataset)} examples")

✅ 45 examples


In [19]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    args=SFTConfig(
        output_dir="/kaggle/working/qwen-unsloth",
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),  # auto-selects
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        save_steps=10,
        save_total_limit=2,
        optim="adamw_8bit",
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        report_to="none",
        dataset_text_field="text",
        max_length=7000,
        packing=False,
    ),
)

trainer.train()

model.save_pretrained("/kaggle/working/qwen-unsloth")
tokenizer.save_pretrained("/kaggle/working/qwen-unsloth")
print("✅ Saved")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/45 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 45 | Num Epochs = 3 | Total steps = 18
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 4,358,144 of 1,548,072,448 (0.28% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,0.494700
10,0.457800
15,0.437100


✅ Saved


## LOAD THE CUSTOM TRAINED MODEL

In [20]:
# ── Load your saved trained model ────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/working/qwen-unsloth",  # ← your saved model
    max_seq_length=7000,
    dtype=torch.float16,
    load_in_4bit=True,
)

# ── Switch to inference mode ──────────────────────────────
FastLanguageModel.for_inference(model)

# ── Prompt ────────────────────────────────────────────────
prompt = "<|im_start|>user\nBuild a single-file with HTML and CSS for a landing page for a travel booking website.<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# ── Generate ──────────────────────────────────────────────
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=7000,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# ── Decode ────────────────────────────────────────────────
generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
response  = generated.split("<|im_start|>assistant")[-1].strip()

# ── Save to HTML ──────────────────────────────────────────
OUTPUT_FILE = "/kaggle/working/new_index.html"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(response)

print(f"✅ Saved to {OUTPUT_FILE}")
print(response)

==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Saved to /kaggle/working/new_index.html
user
Build a single-file with HTML and CSS for a landing page for a travel booking website.
assistant
```html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Travel Booking Website</title>
    <style>
        /* Basic reset */
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Arial', sans-serif;
 


---

## 👨‍💻 About Labellerr's Hands-On Learning in Computer Vision

Thank you for exploring this **Labellerr Hands-On Computer Vision Cookbook**! We hope this notebook helped you learn, prototype, and accelerate your vision projects.  
Labellerr provides ready-to-run Jupyter/Colab notebooks for the latest models and real-world use cases in computer vision, AI agents, and data annotation.

---
## 🧑‍🔬 Check Our Popular Youtube Videos

Whether you're a beginner or a practitioner, our hands-on training videos are perfect for learning custom model building, computer vision techniques, and applied AI:

- [How to Fine-Tune YOLO on Custom Dataset](https://www.youtube.com/watch?v=pBLWOe01QXU)  
  Step-by-step guide to fine-tuning YOLO for real-world use—environment setup, annotation, training, validation, and inference.
- [Build a Real-Time Intrusion Detection System with YOLO](https://www.youtube.com/watch?v=kwQeokYDVcE)  
  Create an AI-powered system to detect intruders in real time using YOLO and computer vision.
- [Finding Athlete Speed Using YOLO](https://www.youtube.com/watch?v=txW0CQe_pw0)  
  Estimate real-time speed of athletes for sports analytics.
- [Object Counting Using AI](https://www.youtube.com/watch?v=smsjBBQcIUQ)  
  Learn dataset curation, annotation, and training for robust object counting AI applications.
---

## 🎦 Popular Labellerr YouTube Videos

Level up your skills and see video walkthroughs of these tools and notebooks on the  
[Labellerr YouTube Channel](https://www.youtube.com/@Labellerr/videos):

- [How I Fixed My Biggest Annotation Nightmare with Labellerr](https://www.youtube.com/watch?v=hlcFdiuz_HI) – Solving complex annotation for ML engineers.
- [Explore Your Dataset with Labellerr's AI](https://www.youtube.com/watch?v=LdbRXYWVyN0) – Auto-tagging, object counting, image descriptions, and dataset exploration.
- [Boost AI Image Annotation 10X with Labellerr's CLIP Mode](https://www.youtube.com/watch?v=pY_o4EvYMz8) – Refine annotations with precision using CLIP mode.
- [Boost Data Annotation Accuracy and Efficiency with Active Learning](https://www.youtube.com/watch?v=lAYu-ewIhTE) – Speed up your annotation workflow using Active Learning.

> 👉 **Subscribe** for Labellerr's deep learning, annotation, and AI tutorials, or watch videos directly alongside notebooks!

---

## 🤝 Stay Connected

- **Website:** [https://www.labellerr.com/](https://www.labellerr.com/)
- **Blog:** [https://www.labellerr.com/blog/](https://www.labellerr.com/blog/)
- **GitHub:** [Labellerr/Hands-On-Learning-in-Computer-Vision](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)
- **LinkedIn:** [Labellerr](https://in.linkedin.com/company/labellerr)
- **Twitter/X:** [@Labellerr1](https://x.com/Labellerr1)

*Happy learning and building with Labellerr!*
